# Semi-Supervised Learning (SSL) - Starter Notebook
SSL combines a small labeled set with a larger unlabeled set to improve model performance.

In [ ]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.semi_supervised import LabelSpreading
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

In [ ]:
X, y = make_classification(n_samples=2500, n_features=20, n_informative=12,
                           n_redundant=2, n_classes=2, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Keep only a small labeled subset
rng = np.random.RandomState(42)
labeled_fraction = 0.1
n_labeled = int(len(X_train) * labeled_fraction)
idx = np.arange(len(X_train))
rng.shuffle(idx)
labeled_idx = idx[:n_labeled]

y_train_partial = -1 * np.ones_like(y_train)
y_train_partial[labeled_idx] = y_train[labeled_idx]
print('Labeled points:', np.sum(y_train_partial != -1), 'Unlabeled points:', np.sum(y_train_partial == -1))

In [ ]:
# Supervised baseline on only labeled subset
rf = RandomForestClassifier(random_state=42)
rf.fit(X_train[labeled_idx], y_train[labeled_idx])
pred_sup = rf.predict(X_test)
acc_sup = accuracy_score(y_test, pred_sup)

# SSL model on full data with missing labels
ssl = LabelSpreading(kernel='rbf', gamma=0.1, alpha=0.2, max_iter=30)
ssl.fit(X_train, y_train_partial)
pred_ssl = ssl.predict(X_test)
acc_ssl = accuracy_score(y_test, pred_ssl)

print(f'Supervised (10% labels) accuracy: {acc_sup:.4f}')
print(f'SSL LabelSpreading accuracy    : {acc_ssl:.4f}')

In [ ]:
# Confidence of pseudo-labeling on train set
probs = ssl.label_distributions_
conf = probs.max(axis=1)
print('Avg confidence over train points:', conf.mean())
print('High-confidence (>0.9) points :', np.sum(conf > 0.9))